In [1]:
# =============================================================================
# Section 1: Factor Generators — SE3 and SE2 factor definitions + C++ codegen
# =============================================================================
import symforce
symforce.set_epsilon_to_symbol()
symforce.set_symbolic_api("symengine")

import symforce.symbolic as sf
from symforce.ops import LieGroupOps
from symforce.opt.noise_models import BarronNoiseModel

# ---- SE3 Factors ----

def between_factor_se3(T_a: sf.Pose3, T_b: sf.Pose3, T_ab_meas: sf.Pose3,
                       sqrt_info_diag: sf.V6, epsilon: sf.Scalar) -> sf.V6:
    """Between (odometry) factor: error = log(T_ab_meas^{-1} * T_a^{-1} * T_b)"""
    error = sf.V6(LieGroupOps.local_coordinates(T_ab_meas, T_a.inverse() * T_b, epsilon=epsilon))
    return error.multiply_elementwise(sqrt_info_diag)

def loop_closure_factor_se3(T_a: sf.Pose3, T_b: sf.Pose3, T_ab_meas: sf.Pose3,
                            sqrt_info_diag: sf.V6, barron_alpha: sf.Scalar,
                            barron_delta: sf.Scalar, epsilon: sf.Scalar) -> sf.V6:
    """Robust loop closure factor with Barron kernel."""
    error = sf.V6(LieGroupOps.local_coordinates(T_ab_meas, T_a.inverse() * T_b, epsilon=epsilon))
    whitened = error.multiply_elementwise(sqrt_info_diag)
    noise_model = BarronNoiseModel(alpha=barron_alpha, scalar_information=1,
                                    x_epsilon=epsilon, delta=barron_delta)
    return noise_model.whiten_norm(whitened, epsilon=epsilon)

def pose_to_landmark_factor_se3(T_b: sf.Pose3, p_b: sf.V3, p_world: sf.V3,
                                 sqrt_info_diag: sf.V3, barron_alpha: sf.Scalar,
                                 barron_delta: sf.Scalar, epsilon: sf.Scalar) -> sf.V3:
    """Robust landmark factor with Barron kernel (SE3)."""
    whitened = (T_b.inverse() * p_world - p_b).multiply_elementwise(sqrt_info_diag)
    noise_model = BarronNoiseModel(alpha=barron_alpha, scalar_information=1,
                                    x_epsilon=epsilon, delta=barron_delta)
    return noise_model.whiten_norm(whitened, epsilon=epsilon)

def prior_se3(T_a: sf.Pose3, T_a_prior: sf.Pose3, sqrt_info_diag: sf.V6,
              epsilon: sf.Scalar) -> sf.V6:
    """Prior factor for SE3."""
    error = sf.V6(LieGroupOps.local_coordinates(T_a_prior, T_a, epsilon=epsilon))
    return error.multiply_elementwise(sqrt_info_diag)

# ---- SE2 Factors ----

def between_factor_se2(T_a: sf.Pose2, T_b: sf.Pose2, T_ab_meas: sf.Pose2,
                       sqrt_info: sf.M33, epsilon: sf.Scalar) -> sf.V3:
    """Between (odometry) factor for SE2."""
    error = sf.V3(LieGroupOps.local_coordinates(T_ab_meas, T_a.inverse() * T_b, epsilon=epsilon))
    return sqrt_info * error

def loop_closure_factor_se2(T_a: sf.Pose2, T_b: sf.Pose2, T_ab_meas: sf.Pose2,
                            sqrt_info: sf.M33, barron_alpha: sf.Scalar,
                            barron_delta: sf.Scalar, epsilon: sf.Scalar) -> sf.V3:
    """Robust loop closure factor for SE2 with Barron kernel."""
    error = sf.V3(LieGroupOps.local_coordinates(T_ab_meas, T_a.inverse() * T_b, epsilon=epsilon))
    whitened = sqrt_info * error
    noise_model = BarronNoiseModel(alpha=barron_alpha, scalar_information=1,
                                    x_epsilon=epsilon, delta=barron_delta)
    return noise_model.whiten_norm(whitened, epsilon=epsilon)

def pose_to_landmark_factor_se2(T_b: sf.Pose2, p_b: sf.V2, p_world: sf.V2,
                                 sqrt_info: sf.M22, barron_alpha: sf.Scalar,
                                 barron_delta: sf.Scalar, epsilon: sf.Scalar) -> sf.V2:
    """Robust landmark factor with Barron kernel (SE2)."""
    error = T_b.inverse() * p_world - p_b
    whitened = sqrt_info * error
    noise_model = BarronNoiseModel(alpha=barron_alpha, scalar_information=1,
                                    x_epsilon=epsilon, delta=barron_delta)
    return noise_model.whiten_norm(whitened, epsilon=epsilon)

def prior_factor_se2(T_a: sf.Pose2, T_a_prior: sf.Pose2, sqrt_info: sf.M33,
                     epsilon: sf.Scalar) -> sf.V3:
    """Prior factor for SE2."""
    error = sf.V3(LieGroupOps.local_coordinates(T_a_prior, T_a, epsilon=epsilon))
    return sqrt_info * error

print("All 8 factors defined (SE3: 4, SE2: 4) — all with Barron kernel where applicable.")

All 8 factors defined (SE3: 4, SE2: 4) — all with Barron kernel where applicable.


In [2]:
# C++ code generation — writes headers to inc/gen/
from symforce.codegen import Codegen, CppConfig
import os

output_dir = os.path.join(os.path.dirname(os.getcwd()), "inc", "gen")

all_specs = [
    # SE3
    (between_factor_se3,            ["T_a", "T_b"]),
    (loop_closure_factor_se3,       ["T_a", "T_b"]),
    (pose_to_landmark_factor_se3,   ["T_b", "p_world"]),
    (prior_se3,                     ["T_a"]),
    # SE2
    (between_factor_se2,            ["T_a", "T_b"]),
    (loop_closure_factor_se2,       ["T_a", "T_b"]),
    (pose_to_landmark_factor_se2,   ["T_b", "p_world"]),
    (prior_factor_se2,              ["T_a"]),
]

for func, which_args in all_specs:
    cg = Codegen.function(func=func, config=CppConfig())
    cg.with_linearization(which_args=which_args).generate_function(
        output_dir=output_dir, skip_directory_nesting=False)
    print(f"  {func.__name__}")

print(f"\nGenerated {len(all_specs)} factor headers in: {output_dir}")

  between_factor_se3
  loop_closure_factor_se3
  pose_to_landmark_factor_se3
  prior_se3
  between_factor_se2
  loop_closure_factor_se2
  pose_to_landmark_factor_se2
  prior_factor_se2

Generated 8 factor headers in: /workspace/1-optimization-and-vis-slam/symforce/inc/gen


In [3]:
# =============================================================================
# Section 2: Demo — Synthetic 4-pose optimization with outlier rejection
# =============================================================================
import numpy as np
from symforce.opt.factor import Factor
from symforce.opt.optimizer import Optimizer
from symforce.values import Values

np.random.seed(42)

# Ground truth: 4 poses and 2 landmarks
gt_poses = [
    sf.Pose3(sf.Rot3.identity(), sf.V3(0, 0, 0)),
    sf.Pose3(sf.Rot3.from_angle_axis(0.1, sf.V3(0, 0, 1)), sf.V3(1, 0.05, 0)),
    sf.Pose3(sf.Rot3.from_angle_axis(0.3, sf.V3(0, 0, 1)), sf.V3(2, 0.2, 0)),
    sf.Pose3(sf.Rot3.from_angle_axis(0.5, sf.V3(0, 0, 1)), sf.V3(3, 0.5, 0)),
]
gt_landmarks = [sf.V3(0.5, 1.0, 0.0), sf.V3(2.5, -0.5, 0.0)]

# Measurements (clean odometry, one landmark OUTLIER, one FALSE loop closure)
odom_meas = [gt_poses[i].inverse() * gt_poses[i+1] for i in range(3)]
lm_obs = [
    (0, 0, gt_poses[0].inverse() * gt_landmarks[0]),
    (1, 0, gt_poses[1].inverse() * gt_landmarks[0]),
    (1, 1, gt_poses[1].inverse() * gt_landmarks[1]),
    (2, 1, gt_poses[2].inverse() * gt_landmarks[1]),
    (3, 1, gt_poses[3].inverse() * gt_landmarks[1] + sf.V3(10.1, -0.1, 0)),  # OUTLIER
]
loop_closures = [
    (3, 0, gt_poses[3].inverse() * gt_poses[0]),                            # correct
    (2, 0, sf.Pose3(sf.Rot3.identity(), sf.V3(50, 50, 0))),                 # FALSE
]

# Build Values with perturbed initial guess
values = Values()
for i, gt in enumerate(gt_poses):
    dR = sf.Rot3.from_angle_axis(np.random.randn() * 0.15, sf.V3(0, 0, 1))
    dt = sf.V3(np.random.randn() * 0.3, np.random.randn() * 0.3, 0)
    values[f"poses[{i}]"] = sf.Pose3(R=dR * gt.R, t=gt.t + dt)
for j, gt_lm in enumerate(gt_landmarks):
    values[f"landmarks[{j}]"] = gt_lm + sf.V3(np.random.randn() * 0.5, np.random.randn() * 0.5, 0)
for i, m in enumerate(odom_meas):
    values[f"odom_meas[{i}]"] = m
for k, (_, _, p_body) in enumerate(lm_obs):
    values[f"lm_obs[{k}]"] = p_body
for c, (_, _, lc_meas) in enumerate(loop_closures):
    values[f"lc_meas[{c}]"] = lc_meas
values["odom_sqrt_info"] = sf.V6.one() * 100
values["prior_sqrt_info"] = sf.V6.one() * 1000
values["lm_sqrt_info"] = sf.V3.one() * 50
values["lc_sqrt_info"] = sf.V6.one() * 50
values["prior_pose"] = gt_poses[0]
values["barron_alpha"] = -2.0   # Geman-McClure
values["barron_delta"] = 1.0
values["epsilon"] = sf.numeric_epsilon

# Build factor graph
factors = [
    Factor(residual=prior_se3, keys=["poses[0]", "prior_pose", "prior_sqrt_info", "epsilon"]),
]
for i in range(3):
    factors.append(Factor(residual=between_factor_se3,
        keys=[f"poses[{i}]", f"poses[{i+1}]", f"odom_meas[{i}]", "odom_sqrt_info", "epsilon"]))
for k, (pi, li, _) in enumerate(lm_obs):
    factors.append(Factor(residual=pose_to_landmark_factor_se3,
        keys=[f"poses[{pi}]", f"lm_obs[{k}]", f"landmarks[{li}]",
              "lm_sqrt_info", "barron_alpha", "barron_delta", "epsilon"]))
for c, (ci, cj, _) in enumerate(loop_closures):
    factors.append(Factor(residual=loop_closure_factor_se3,
        keys=[f"poses[{ci}]", f"poses[{cj}]", f"lc_meas[{c}]",
              "lc_sqrt_info", "barron_alpha", "barron_delta", "epsilon"]))

opt_keys = [f"poses[{i}]" for i in range(4)] + [f"landmarks[{j}]" for j in range(2)]
optimizer = Optimizer(factors=factors, optimized_keys=opt_keys,
    params=Optimizer.Params(verbose=False, initial_lambda=1e4, lambda_down_factor=0.5,
                            iterations=50, early_exit_min_reduction=1e-8))
result = optimizer.optimize(values)

# Report
print(f"Status: {result.status}  |  Final error: {float(result.error()):.2e}")
for i, gt in enumerate(gt_poses):
    opt_t = [float(x) for x in result.optimized_values[f"poses[{i}]"].position()]
    gt_t = [float(x) for x in gt.t]
    err = sum((a - b) ** 2 for a, b in zip(opt_t, gt_t)) ** 0.5
    print(f"  Pose {i}: t_err = {err:.2e}")
for j, gt_lm in enumerate(gt_landmarks):
    opt_l = [float(x) for x in result.optimized_values[f"landmarks[{j}]"]]
    gt_l = [float(x) for x in gt_lm]
    err = sum((a - b) ** 2 for a, b in zip(opt_l, gt_l)) ** 0.5
    print(f"  Landmark {j}: err = {err:.2e}")
print("\nBarron kernel successfully rejects the outlier landmark and false loop closure.")

Status: optimization_status_t.SUCCESS  |  Final error: 4.00e+00
  Pose 0: t_err = 4.37e-13
  Pose 1: t_err = 1.18e-10
  Pose 2: t_err = 2.02e-10
  Pose 3: t_err = 7.39e-10
  Landmark 0: err = 6.19e-11
  Landmark 1: err = 1.17e-09

Barron kernel successfully rejects the outlier landmark and false loop closure.
